In [34]:
import os
import json
import sqlite3
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [35]:
load_dotenv(override=True)

groq_api_key = os.getenv('GROQ_API_KEY')
if groq_api_key:
    print(f"GROQ API Key exists and begins {groq_api_key[:4]}")
else:
    print("GROQ API Key not set")
    
MODEL = "openai/gpt-oss-120b"
groq = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_api_key)

GROQ API Key exists and begins gsk_


In [36]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [37]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [38]:
# There's a particular dictionary structure that's required to describe our function:

get_price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

set_price_function = {
    "name": "set_ticket_price",
    "description": "Set the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the admin wants to set the price for",
            },
            "price": {
                "type": "number",
                "description": "The price of the return ticket",
            }
        },
        "required": ["destination_city", "price"],
        "additionalProperties": False
    }
}

In [39]:
tools = [{"type": "function", "function": get_price_function}, {"type": "function", "function": set_price_function}]

In [40]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [41]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [42]:
available_functions = {
    "get_ticket_price": get_ticket_price,
    "set_ticket_price": set_ticket_price
}

In [43]:
def handle_tool_calls(message):
    print(message)
    responses = []

    for tool_call in message.tool_calls:

        function_name = tool_call.function.name
        function_to_call = available_functions.get(function_name)

        if not function_to_call:
            responses.append({
                "role": "tool",
                "content": f"Error: Function {function_name} not found",
                "tool_call_id": tool_call.id
            })
            continue

        arguments = json.loads(tool_call.function.arguments)

        if function_name == "get_ticket_price":
            city = arguments.get('destination_city')
            price_details = function_to_call(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })

        elif function_name == "set_ticket_price":
            city = arguments.get('destination_city')
            price = arguments.get('price')
            function_to_call(city, price)
            responses.append({
                "role": "tool",
                "content": "Ticket price updated successfully",
                "tool_call_id": tool_call.id
            })
    return responses

In [44]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = groq.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = groq.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.


ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='fc_683cfd88-dbfb-4d19-9c93-0d587593ae47', function=Function(arguments='{"destination_city":"Berlin","price":1100}', name='set_ticket_price'), type='function')], reasoning='User wants to update flight ticket prices for Berlin $1100, Tokyo $850, Singapore $400. Need to call set_ticket_price three times.')
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='fc_8da5bd7a-fc84-48a0-86ff-90cf1f320069', function=Function(arguments='{"destination_city":"Tokyo","price":850}', name='set_ticket_price'), type='function')], reasoning='Now set Tokyo.')
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolC